<a href="https://colab.research.google.com/github/Prasadn1234/DataScience/blob/claude/teaching-integers-8z62ib/Capstone_MultiAgent_Car_Manufacturing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project: Multi-Agent AI for Car Manufacturing — Marketing, Sales & Service

## Goal

Design and prototype a **multi-agent AI system** that supports an automotive manufacturer/dealer group across three functions that today run in silos:

- **Marketing** — campaign targeting, content generation, lead generation
- **Sales** — lead qualification, test-drive booking, financing quotes, negotiation support
- **Service** — appointment scheduling, diagnostics triage, recall/warranty lookup, parts availability

The system gives every customer a **single continuous relationship** with the brand: a lead nurtured by Marketing hands off cleanly to Sales, and the same customer profile follows them into Service after purchase — instead of three disconnected chatbots that each start from zero.

This notebook ties together patterns you've already built in this repo:
- Function-calling agents (`CRM_Lead_Qualifier_Agent.ipynb`)
- RAG / vector stores for grounded answers (`Vector_Store_Notebook_04272026.ipynb`)
- Structured outputs via Pydantic (TODO 4.1 clinical agent work)
- RAG evaluation practices (`Support_Bot_Evaluation_04272026.ipynb`)

into one coherent, end-to-end capstone.

## 1. Business Context & Success Metrics

| Pain point today | What the multi-agent system fixes |
|---|---|
| Marketing leads get emailed to a generic sales inbox and go cold | Marketing Agent hands off a structured, scored lead directly into the Sales Agent's queue |
| Sales reps re-ask questions the customer already answered in Marketing | Shared **Customer-360 memory** persists context across agents |
| Service advisors can't see purchase/financing history | Service Agent reads the same customer record Sales created |
| Chatbots hallucinate on recalls/warranty terms | Service & Sales agents are grounded via RAG over manuals/policy docs, with a guardrail/evaluation layer |

**Target metrics for the capstone:**
- Lead-to-qualified-lead conversion time (minutes, not days)
- % of service answers grounded in retrieved source docs (no hallucinated recall/warranty claims)
- Handoff completeness: % of fields carried over between agents without re-asking the customer
- Latency per agent turn and end-to-end scenario

## 2. System Architecture

```
                         ┌───────────────────────┐
                         │   Orchestrator Agent   │
                         │  (intent routing +     │
                         │   Customer-360 memory) │
                         └───────────┬────────────┘
              ┌───────────────┬──────┴───────┬───────────────┐
              ▼               ▼              ▼               ▼
     ┌─────────────┐  ┌──────────────┐ ┌──────────────┐ ┌───────────────┐
     │  Marketing   │  │    Sales     │ │   Service    │ │  Knowledge/RAG │
     │    Agent     │  │    Agent     │ │    Agent     │ │     Agent      │
     └──────┬───────┘  └──────┬───────┘ └──────┬───────┘ └────────┬───────┘
            │                 │                │                  │
     campaign/segment   lead scoring,    scheduling, recall   manuals, FAQs,
     tools, content gen financing tools  triage, parts lookup policy docs (vector store)

                         ┌───────────────────────┐
                         │  Evaluation/Guardrail  │
                         │  Agent (safety, PII,   │
                         │  groundedness checks)  │
                         └───────────────────────┘
```

**Orchestrator Agent** — the only agent the customer/user directly talks to. It classifies intent ("I want a quote" vs. "my check-engine light is on"), pulls/updates the shared Customer-360 record, and routes to the right specialist agent via function calling — the same mechanism used in `CRM_Lead_Qualifier_Agent.ipynb`, just with one dispatch layer added on top.

**Specialist agents** each own a narrow set of tools and a system prompt scoped to their job — this keeps prompts small and testable instead of one giant do-everything prompt.

**Knowledge/RAG Agent** is shared infrastructure: owner's manuals, warranty policy, recall bulletins, and financing FAQs live in a vector store (as in `Vector_Store_Notebook_04272026.ipynb`); Sales and Service agents both query it instead of guessing.

**Evaluation/Guardrail Agent** runs after every specialist response — checks for hallucination on safety-critical claims (recalls, warranty), PII leakage, and tone — following the evaluation criteria from `Support_Bot_Evaluation_04272026.ipynb`.

## 3. Structured Data Contracts

Agents hand off to each other through **typed objects, not free text** — this is what makes the handoff reliable. Pydantic models define the shared "Customer-360" record and each agent's output shape (same approach as the clinical agent's structured outputs).

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal
from datetime import date


class CustomerProfile(BaseModel):
    """Shared Customer-360 record every agent reads from and writes to."""
    customer_id: str
    name: str
    email: str
    phone: Optional[str] = None
    stage: Literal["marketing_lead", "qualified_lead", "customer", "service_customer"] = "marketing_lead"
    interested_model: Optional[str] = None
    owned_vin: Optional[str] = None
    notes: list[str] = Field(default_factory=list)


class LeadScore(BaseModel):
    """Output of the Marketing Agent, consumed by the Sales Agent."""
    customer_id: str
    score: int = Field(ge=0, le=100)
    reason: str
    recommended_model: str
    campaign_source: str


class FinancingQuote(BaseModel):
    """Output of the Sales Agent."""
    customer_id: str
    model: str
    monthly_payment_usd: float
    term_months: int
    apr_percent: float


class ServiceTicket(BaseModel):
    """Output of the Service Agent."""
    customer_id: str
    vin: str
    issue_summary: str
    severity: Literal["low", "medium", "high", "safety_recall"]
    recommended_appointment: Optional[date] = None
    grounded_in_docs: bool = Field(description="True if the diagnosis cites a retrieved manual/recall source")


print("Schemas ready:", [m.__name__ for m in (CustomerProfile, LeadScore, FinancingQuote, ServiceTicket)])

## 4. Agent Tools (mocked business logic)

As in `CRM_Lead_Qualifier_Agent.ipynb`, we mock the systems of record (CRM, DMS, parts inventory) with plain dictionaries/functions. In production these would be real API calls to the dealer CRM, DMS (dealer management system), and parts inventory.

In [ ]:
# --- Mock "systems of record" ---

_CUSTOMER_DB: dict[str, CustomerProfile] = {}

_RECALL_DB = {
    "Model X": ["Recall 2025-114: fuel pump software update"],
    "Model Y": [],
}

_PARTS_INVENTORY = {
    "brake_pad_modelx": 12,
    "fuel_pump_modelx": 0,
}


def get_or_create_customer(customer_id: str, name: str, email: str) -> CustomerProfile:
    if customer_id not in _CUSTOMER_DB:
        _CUSTOMER_DB[customer_id] = CustomerProfile(customer_id=customer_id, name=name, email=email)
    return _CUSTOMER_DB[customer_id]


# --- Marketing Agent tool ---
def score_lead(customer: CustomerProfile, campaign_source: str, browsed_model: str) -> LeadScore:
    score = 70 if campaign_source == "test_drive_signup" else 40
    customer.stage = "qualified_lead" if score >= 60 else "marketing_lead"
    customer.interested_model = browsed_model
    return LeadScore(
        customer_id=customer.customer_id,
        score=score,
        reason=f"Source={campaign_source}",
        recommended_model=browsed_model,
        campaign_source=campaign_source,
    )


# --- Sales Agent tool ---
def generate_financing_quote(customer: CustomerProfile, model: str, term_months: int = 60) -> FinancingQuote:
    base_price = {"Model X": 45000, "Model Y": 38000}.get(model, 40000)
    apr = 4.9
    monthly = round((base_price * (1 + apr / 100)) / term_months, 2)
    customer.stage = "customer"
    return FinancingQuote(customer_id=customer.customer_id, model=model, monthly_payment_usd=monthly,
                           term_months=term_months, apr_percent=apr)


# --- Service Agent tools ---
def check_recalls(model: str) -> list[str]:
    return _RECALL_DB.get(model, [])


def check_parts_availability(part_key: str) -> int:
    return _PARTS_INVENTORY.get(part_key, 0)


def open_service_ticket(customer: CustomerProfile, vin: str, issue: str, model: str) -> ServiceTicket:
    recalls = check_recalls(model)
    severity = "safety_recall" if recalls else "medium"
    customer.owned_vin = vin
    customer.stage = "service_customer"
    return ServiceTicket(customer_id=customer.customer_id, vin=vin, issue_summary=issue,
                          severity=severity, grounded_in_docs=True)


print("Tools defined for Marketing, Sales, and Service agents.")

## 5. Orchestrator: Intent Routing

The Orchestrator classifies the incoming message and dispatches to the right specialist. Below is a lightweight rule-based router so the notebook runs with **no API key required**. In the next cell, swap `classify_intent` for an LLM function-calling classifier (same pattern as `CRM_Lead_Qualifier_Agent.ipynb`) — the rest of the orchestration logic doesn't change, because the router only ever hands off typed objects.

In [ ]:
def classify_intent(message: str) -> str:
    """Rule-based stand-in for an LLM function-calling classifier."""
    m = message.lower()
    if any(k in m for k in ["test drive", "looking at", "interested in"]):
        return "marketing"
    if any(k in m for k in ["financing", "monthly payment", "buy", "quote"]):
        return "sales"
    if any(k in m for k in ["check engine", "noise", "appointment", "recall", "repair"]):
        return "service"
    return "unknown"


def orchestrate(customer_id: str, name: str, email: str, message: str, **kwargs):
    customer = get_or_create_customer(customer_id, name, email)
    intent = classify_intent(message)

    if intent == "marketing":
        result = score_lead(customer, kwargs.get("campaign_source", "organic"), kwargs["model"])
    elif intent == "sales":
        result = generate_financing_quote(customer, kwargs["model"])
    elif intent == "service":
        result = open_service_ticket(customer, kwargs["vin"], message, kwargs["model"])
    else:
        result = "Sorry, I couldn't determine intent — routing to a human agent."

    return intent, result, customer


# --- End-to-end scenario: same customer, three touchpoints over time ---
intent, result, cust = orchestrate("cust_001", "Jane Doe", "jane@example.com",
                                    "I'm interested in the Model X, can I book a test drive?",
                                    campaign_source="test_drive_signup", model="Model X")
print(f"[{intent}] -> {result}\n")

intent, result, cust = orchestrate("cust_001", "Jane Doe", "jane@example.com",
                                    "What would my monthly payment be?", model="Model X")
print(f"[{intent}] -> {result}\n")

intent, result, cust = orchestrate("cust_001", "Jane Doe", "jane@example.com",
                                    "My check engine light came on, can I get an appointment?",
                                    vin="1HGCM82633A004352", model="Model X")
print(f"[{intent}] -> {result}\n")

print("Final Customer-360 record:")
print(cust.model_dump_json(indent=2))

Notice the last ticket comes back `severity="safety_recall"` — Jane's Model X has an open recall (fuel pump), and the Service Agent surfaces that automatically because it checked `_RECALL_DB` before answering, rather than just describing the symptom. Also notice the **same `customer_id`** carried a name, model interest, and eventually a VIN across all three calls — no re-asking.

## 6. Swapping in Real LLM Agents + RAG

To move from this scaffold to production-grade agents:

1. **Replace `classify_intent`** with an OpenAI function-calling classifier, exactly like the tool-selection loop in `CRM_Lead_Qualifier_Agent.ipynb` — the model picks one of `route_marketing`, `route_sales`, `route_service` as its "tool call".
2. **Ground Service & Sales answers in a vector store**: load owner's manuals, warranty policy PDFs, and recall bulletins into a Chroma store (same setup as `Vector_Store_Notebook_04272026.ipynb`), and have `open_service_ticket` retrieve relevant chunks before the LLM drafts `issue_summary`, so `grounded_in_docs` reflects a real citation instead of a hardcoded `True`.
3. **Add the Evaluation/Guardrail Agent** as a post-processing step using the correctness/groundedness rubric from `Support_Bot_Evaluation_04272026.ipynb` — run it against every Service Agent response before it reaches the customer, since a hallucinated recall/warranty claim is the highest-severity failure mode in this domain.
4. **Persist `CustomerProfile`** to a real database instead of the in-memory `_CUSTOMER_DB` dict.

## 7. Tech Stack Summary

| Layer | Suggested tool |
|---|---|
| LLM + function calling | OpenAI API (as in `CRM_Lead_Qualifier_Agent.ipynb`) |
| Multi-agent orchestration | LangGraph or a hand-rolled router (shown above) — either works for this scope |
| Structured I/O | Pydantic |
| Knowledge base / RAG | LangChain + Chroma (as in `Vector_Store_Notebook_04272026.ipynb`) |
| Evaluation | LangSmith custom evaluators (as in `Support_Bot_Evaluation_04272026.ipynb`) |
| Systems of record (mocked here) | CRM, DMS, parts inventory APIs |

## 8. Risks & Mitigations

| Risk | Mitigation |
|---|---|
| Hallucinated recall/warranty guidance (safety-critical) | Force retrieval-grounded answers; block any Service response that isn't backed by a retrieved doc for recall/warranty claims |
| PII/financial data (financing quotes, contact info) leaking across agents or logs | Scope each agent's tool access to only the fields it needs; redact PII in evaluation logs |
| Customer re-asked the same question by multiple agents | Shared Customer-360 record is the single source of truth; orchestrator always reads/writes it first |
| Agent scope creep (Sales Agent trying to diagnose mechanical issues) | Narrow system prompts + tool allowlists per agent; Orchestrator enforces routing, not the specialist agents themselves |

## 9. Capstone Deliverables Checklist

- [ ] Architecture diagram + written design doc (this notebook, section 2)
- [ ] Pydantic schemas for all cross-agent handoffs
- [ ] Working orchestrator with at least 3 specialist agents (Marketing, Sales, Service)
- [ ] RAG-grounded Service Agent with citations
- [ ] Evaluation harness scoring correctness/groundedness/safety on a test set of ~20 scenarios
- [ ] End-to-end demo: one customer moving from lead → sale → service
- [ ] Write-up of failure modes observed and how the guardrail agent caught (or missed) them